In [ ]:
# Step 1
# Install bleeding edge dependencies transformers, PEFT to support gemma4 configurations and trl
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q git+https://github.com/huggingface/peft.git
!pip install -q git+https://github.com/huggingface/trl.git
!pip install -q accelerate bitsandbytes datasets numpy wandb

In [ ]:
# Step 2
# Imports and Hyperparameters
import torch
import numpy as np
import wandb
import os
from google.colab import userdata
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

MODEL_ID = "google/gemma-4-e4b"
LORA_RANK = 16
LORA_ALPHA = 32
POPULATION_SIZE = 16
LEARNING_RATE = 0.005  # Reduced to prevent weight collapse
NOISE_STD = 0.005      # Reduced for finer exploration
GENERATIONS = 50
BATCH_SIZE = 4

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# Step 3
# Load Base Model in 4-bit precision
# Inject LoRA Adapters
from transformers import BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map={"": 0},
    quantization_config=bnb_config,
    dtype=torch.float16
)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj.linear", "v_proj.linear", "k_proj.linear", "o_proj.linear"],
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

essa_params = []

def extract_and_reparameterize(module, rank):
    device = module.lora_A.default.weight.device

    # IMPROVEMENT: Initialize B to non-zero to prevent zero singular values
    torch.nn.init.normal_(module.lora_B.default.weight, mean=0.0, std=0.02)

    A = module.lora_A.default.weight.data.float().cpu()
    B = module.lora_B.default.weight.data.float().cpu()

    W = B @ A
    # IMPROVEMENT: Use non-deprecated linalg.svd
    U, S, Vh = torch.linalg.svd(W, full_matrices=False)
    V = Vh.T

    U_r = U[:, :rank].to(device)
    S_r = S[:rank].to(device)
    V_r = V[:, :rank].to(device)

    # IMPROVEMENT: Removed unnecessary gradient tracking (S_r.requires_grad = True)
    return {"module": module, "U": U_r, "S": S_r, "V": V_r}

def apply_singular_values(param_dict, current_S):
    U, V, module = param_dict["U"], param_dict["V"], param_dict["module"]
    S_safe = torch.relu(current_S) + 1e-8
    S_sqrt = torch.diag(torch.sqrt(S_safe))

    module.lora_B.default.weight.data = (U @ S_sqrt).to(module.lora_B.default.weight.dtype)
    module.lora_A.default.weight.data = (S_sqrt @ V.T).to(module.lora_A.default.weight.dtype)

for name, module in model.named_modules():
    if hasattr(module, "lora_A") and hasattr(module, "lora_B"):
        essa_params.append(extract_and_reparameterize(module, LORA_RANK))

In [ ]:
# Step 4
import logging
from datasets import load_dataset
from transformers import pipeline
import numpy as np

# Suppress sequential pipeline warnings
logging.getLogger("transformers.pipelines.base").setLevel(logging.ERROR)

dataset = load_dataset("Maximofn/short-jokes-dataset", split="train[:1000]")

def get_batch(batch_size):
    indices = np.random.choice(len(dataset), batch_size, replace=False)
    return [dataset[int(i)] for i in indices]

reward_model = pipeline(
    "text-classification",
    model="OpenAssistant/reward-model-deberta-v3-base",
    device=0
)

def evaluate_humor_reward_batch(responses, verbose=True):
    safe_responses = []
    for resp in responses:
        # Prevent empty strings and excessive repetitions from crashing the evaluator
        clean_resp = resp.strip()[:512]
        if not clean_resp:
            clean_resp = "empty"
        safe_responses.append(clean_resp)

    # Force batch processing parameter
    results = reward_model(safe_responses, truncation=True, batch_size=len(safe_responses))

    scores = []
    for i, (response, result) in enumerate(zip(responses, results)):
        if not response.strip() or response == "empty":
            score = -10.0 # Heavy penalty for empty generations
        else:
            score = result['score']
            if len(response) < 10:
                score -= 2.0 # Heavier length penalty

        if verbose:
            print(f"--- Response {i+1} ---\nText: {response.strip()}\nScore: {score:.4f}\n")

        scores.append(score)

    return scores

In [ ]:
# Step 5
# Batched Reward ESSA Loop
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
wandb.login()
wandb.init(
    project="essa-humor-finetune",
    config={
        "lora_rank": LORA_RANK,
        "population_size": POPULATION_SIZE,
        "learning_rate": LEARNING_RATE,
        "noise_std": NOISE_STD,
        "batch_size": BATCH_SIZE
    }
)

# Persistent table for tracking across generations
sample_table = wandb.Table(columns=["Generation", "Prompt", "Best Output", "Reward"])

eval_batch = get_batch(BATCH_SIZE)

# Use native chat template for correct tokenization
prompts = []
for sample in eval_batch:
    chat = [
        {"role": "user", "content": f"You are a comedian. Make this joke funnier: {sample.get('Joke', 'Tell me a joke.')}"}
    ]
    prompts.append(tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True))

inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to("cuda")
prompt_lengths = inputs.input_ids.shape[1]

gen_kwargs = {
    "max_new_tokens": 64,
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.9,
    "repetition_penalty": 1.15,
    "pad_token_id": tokenizer.pad_token_id,
    "attention_mask": inputs.attention_mask
}

for generation in range(GENERATIONS):
    population_noise = [[torch.randn_like(p["S"]) * NOISE_STD for p in essa_params] for _ in range(POPULATION_SIZE)]
    rewards = np.zeros(POPULATION_SIZE)

    for param in essa_params:
        apply_singular_values(param, param["S"])

    with torch.no_grad():
        base_outputs = model.generate(inputs.input_ids, **gen_kwargs)
    base_responses = tokenizer.batch_decode(base_outputs[:, prompt_lengths:], skip_special_tokens=True)
    baseline_reward = np.mean(evaluate_humor_reward_batch(base_responses, verbose=False))

    all_generation_responses = []

    for p_idx, noise in enumerate(population_noise):
        for i, param in enumerate(essa_params):
            apply_singular_values(param, param["S"] + noise[i])

        with torch.no_grad():
            outputs = model.generate(inputs.input_ids, **gen_kwargs)

        generated_tokens = outputs[:, prompt_lengths:]
        responses = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        all_generation_responses.extend(responses)

        batch_rewards = evaluate_humor_reward_batch(responses, verbose=False)
        rewards[p_idx] = np.mean(batch_rewards)

    normalized_rewards = rewards - baseline_reward

    update_magnitude = 0.0
    with torch.no_grad():
        for i, param in enumerate(essa_params):
            grad = sum(normalized_rewards[p_idx] * population_noise[p_idx][i] for p_idx in range(POPULATION_SIZE))

            if torch.isnan(grad).any() or torch.isinf(grad).any():
                continue

            grad_step = LEARNING_RATE * (grad / (POPULATION_SIZE * NOISE_STD))

            # Implement gradient clipping to prevent extreme matrix distortions
            grad_step = torch.clamp(grad_step, min=-0.05, max=0.05)

            param["S"].data += grad_step
            update_magnitude += grad_step.norm().item()
            apply_singular_values(param, param["S"])

    update_magnitude /= len(essa_params)

    mean_reward = np.mean(rewards)
    max_reward = np.max(rewards)
    reward_variance = np.var(rewards)
    avg_length = np.mean([len(resp.split()) for resp in all_generation_responses])

    print(f"Gen {generation + 1}/{GENERATIONS} | Base: {baseline_reward:.4f} | Max: {max_reward:.4f} | Mean: {mean_reward:.4f}")

    wandb.log({
        "generation": generation + 1,
        "baseline_reward": baseline_reward,
        "mean_reward": mean_reward,
        "max_reward": max_reward,
        "reward_variance": reward_variance,
        "update_magnitude": update_magnitude,
        "avg_response_length": avg_length
    })

    if generation % 5 == 0:
        best_idx = np.argmax(rewards)
        best_batch_responses = all_generation_responses[best_idx * BATCH_SIZE : (best_idx + 1) * BATCH_SIZE]
        # Append data to persistent table
        sample_table.add_data(generation + 1, prompts[0], best_batch_responses[0], max_reward)

# Log the accumulated table once at the end of the run
wandb.log({"generation_samples": sample_table})
wandb.finish()

In [ ]:
# Save the final LoRA adapters
# IMPROVEMENT: Corrected output directory to reflect the joke dataset
output_dir = "./gemma4-e4b-essa-jokes"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"ESSA optimized adapters saved to {output_dir}")